In [1]:
from ultralytics import YOLO

# 加载教师模型

In [2]:


# 加载模型
teacher = YOLO('weights/multi_yolo11s.pt')  # 或 yolo11.yaml
teacher.eval().cuda()
for param in teacher.parameters():
    param.requires_grad = False  # 冻结教师模型



# 准备学生模型

### 加载原始YOLO11n模型

In [3]:
# student_original = YOLO(cfg='yolo11n.yaml', ch=3, nc=num_classes).cuda()
# student_original.load_state_dict(torch.load('yolo11n_pretrained.pth'))

In [4]:
student = YOLO('yolo11n.yaml')

## 蒸馏损失设计

### 输出层蒸馏

In [8]:
# 使用教师和学生的 预测 logits（obj + cls + bbox）
# 对分类使用 KL 散度，对 bbox 使用 L2 或 GIoU loss 加权
def distill_output_loss(student_pred, teacher_pred, T=4.0):
    # 分离 obj, cls, bbox
    s_obj, s_cls, s_box = student_pred
    t_obj, t_cls, t_box = teacher_pred

    # 分类蒸馏（KL Loss）
    kl_loss = F.kl_div(
        F.log_softmax(s_cls / T, dim=-1),
        F.softmax(t_cls / T, dim=-1),
        reduction='batchmean'
    ) * (T ** 2)

    # 边界框回归（L2）
    box_loss = F.mse_loss(s_box, t_box)

    # objectness 蒸馏
    obj_loss = F.mse_loss(s_obj, t_obj)

    return kl_loss + 0.1 * box_loss + 0.1 * obj_loss

### 中间层蒸馏

In [9]:
# 选择 backbone 或 neck 中的关键层（如 P3, P4, P5）
# 使用 L2 loss 或 Attention-based feature distillation
def distill_feature_loss(feat_s, feat_t):
    # 假设 feat_s 和 feat_t 形状不同 → 需适配
    if feat_s.shape != feat_t.shape:
        adapter = nn.Conv2d(feat_s.shape[1], feat_t.shape[1], 1).cuda()
        feat_s = adapter(feat_s)
    return F.mse_loss(feat_s, feat_t)

## 提取中间特征（Hook 机制）

In [10]:
features_teacher = {}
features_student = {}

def hook_fn(name, feat_dict):
    def hook(module, input, output):
        feat_dict[name] = output
    return hook

# 注册 hook（根据你的模型结构选择层名）
for name in ['backbone.stage3', 'neck.P4', 'neck.P5']:
    teacher.get_submodule(name).register_forward_hook(hook_fn(name, features_teacher))
    student.get_submodule(name).register_forward_hook(hook_fn(name, features_student))

AttributeError: YOLO has no attribute `backbone`

## 蒸馏训练循环

In [13]:
import torch

optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)
criterion_ce = torch.nn.CrossEntropyLoss()  # 原始任务损失（可选）
lambda_feat = 1.0
lambda_out = 1.0
lambda_task = 0.5  # 如果保留原始监督信号
epochs = 10


for epoch in range(epochs):
    for imgs, targets in dataloader:
        imgs = imgs.cuda()
        targets = targets.cuda()

        # 清空特征字典
        features_teacher.clear()
        features_student.clear()

        with torch.no_grad():
            _ = teacher(imgs)  # 触发 hook，填充 features_teacher

        pred_s = student(imgs)  # 触发 hook，填充 features_student

        # 1. 原始检测损失（可选）
        loss_task = compute_yolo_loss(pred_s, targets)

        # 2. 输出蒸馏损失
        loss_out = distill_output_loss(pred_s, teacher_pred=teacher(imgs))

        # # 3. 中间层蒸馏损失
        # loss_feat = 0
        # for key in features_teacher.keys():
        #     loss_feat += distill_feature_loss(features_student[key], features_teacher[key])
        # loss_feat /= len(features_teacher)

        total_loss = lambda_task * loss_task + lambda_out * loss_out # + lambda_feat * loss_feat

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

NameError: name 'dataloader' is not defined

# 知识蒸馏基类

In [14]:

# 3_kd_base.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt

class BaseDistiller:
    """知识蒸馏基类"""
    def __init__(self, teacher, student, temperature=4.0, alpha=0.5):
        self.teacher = teacher
        self.student = student
        self.temperature = temperature
        self.alpha = alpha
        
        # 冻结教师模型参数
        for param in self.teacher.parameters():
            param.requires_grad = False
            
        self.teacher.eval()
        self.student.train()
        
    def compute_kd_loss(self, student_logits, teacher_logits, labels):
        """计算知识蒸馏损失"""
        raise NotImplementedError
        
    def train_epoch(self, dataloader, optimizer, criterion):
        """训练一个epoch"""
        self.student.train()
        total_loss = 0
        
        for batch_idx, (images, labels) in enumerate(tqdm(dataloader)):
            images = images.cuda()
            labels = labels.cuda()
            
            # 前向传播
            with torch.no_grad():
                teacher_outputs = self.teacher(images)
            
            student_outputs = self.student(images)
            
            # 计算损失
            loss = self.compute_kd_loss(
                student_outputs, 
                teacher_outputs, 
                labels
            )
            
            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        return total_loss / len(dataloader)
    
    def validate(self, dataloader):
        """验证模型性能"""
        self.student.eval()
        total_correct = 0
        total_samples = 0
        
        with torch.no_grad():
            for images, labels in dataloader:
                images = images.cuda()
                labels = labels.cuda()
                
                outputs = self.student(images)
                _, predicted = torch.max(outputs, 1)
                
                total_correct += (predicted == labels).sum().item()
                total_samples += labels.size(0)
                
        accuracy = total_correct / total_samples
        return accuracy
    
    def train(self, train_loader, val_loader, epochs, lr=0.001):
        """完整训练过程"""
        optimizer = optim.Adam(self.student.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        
        train_losses = []
        val_accuracies = []
        
        print(f"开始知识蒸馏训练，共 {epochs} 个epochs")
        
        for epoch in range(epochs):
            # 训练
            train_loss = self.train_epoch(train_loader, optimizer, criterion)
            train_losses.append(train_loss)
            
            # 验证
            val_acc = self.validate(val_loader)
            val_accuracies.append(val_acc)
            
            print(f"Epoch [{epoch+1}/{epochs}] - "
                  f"Train Loss: {train_loss:.4f}, "
                  f"Val Acc: {val_acc:.4f}")
            
            # 保存检查点
            if (epoch + 1) % 10 == 0:
                self.save_checkpoint(epoch, train_loss, val_acc)
        
        # 绘制训练曲线
        self.plot_training_curves(train_losses, val_accuracies)
        
        return train_losses, val_accuracies
    
    def save_checkpoint(self, epoch, loss, accuracy):
        """保存检查点"""
        checkpoint = {
            'epoch': epoch,
            'student_state_dict': self.student.state_dict(),
            'loss': loss,
            'accuracy': accuracy
        }
        torch.save(checkpoint, f'checkpoint_epoch_{epoch}.pth')
        print(f"检查点已保存: checkpoint_epoch_{epoch}.pth")
    
    def plot_training_curves(self, train_losses, val_accuracies):
        """绘制训练曲线"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        
        ax1.plot(train_losses, label='Train Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title('Training Loss')
        ax1.legend()
        ax1.grid(True)
        
        ax2.plot(val_accuracies, label='Val Accuracy', color='orange')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.set_title('Validation Accuracy')
        ax2.legend()
        ax2.grid(True)
        
        plt.tight_layout()
        plt.savefig('training_curves.png', dpi=150)
        plt.show()

# 三种蒸馏方式

In [16]:
# 4_distillation_methods.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeatureDistiller(BaseDistiller):
    """纯中间层特征蒸馏"""
    def __init__(self, teacher, student, temperature=4.0, alpha=0.7):
        super().__init__(teacher, student, temperature, alpha)
        
    def compute_kd_loss(self, student_features, teacher_features, labels):
        """
        计算特征蒸馏损失
        student_features: 学生模型特征列表 [feat1, feat2, ...]
        teacher_features: 教师模型特征列表 [feat1, feat2, ...]
        """
        # 特征对齐损失（使用MSE或余弦相似度）
        feature_loss = 0
        for s_feat, t_feat in zip(student_features, teacher_features):
            # 调整特征图尺寸（如果需要）
            if s_feat.shape != t_feat.shape:
                t_feat = F.adaptive_avg_pool2d(t_feat, s_feat.shape[2:])
            
            # 计算特征损失（可以使用多种损失函数组合）
            mse_loss = F.mse_loss(s_feat, t_feat)
            cosine_loss = 1 - F.cosine_similarity(
                s_feat.flatten(1), 
                t_feat.flatten(1)
            ).mean()
            
            feature_loss += mse_loss + 0.1 * cosine_loss
        
        return feature_loss

class OutputDistiller(BaseDistiller):
    """纯输出层蒸馏（传统KD）"""
    def __init__(self, teacher, student, temperature=4.0, alpha=0.5):
        super().__init__(teacher, student, temperature, alpha)
        self.criterion_ce = nn.CrossEntropyLoss()
        self.criterion_kl = nn.KLDivLoss(reduction='batchmean')
        
    def compute_kd_loss(self, student_logits, teacher_logits, labels):
        """
        计算输出层蒸馏损失
        使用KL散度 + 交叉熵
        """
        # 软化教师输出
        soft_teacher = F.softmax(teacher_logits / self.temperature, dim=1)
        soft_student = F.log_softmax(student_logits / self.temperature, dim=1)
        
        # KL散度损失（蒸馏损失）
        kd_loss = self.criterion_kl(soft_student, soft_teacher) * (self.temperature ** 2)
        
        # 学生与真实标签的交叉熵损失
        ce_loss = self.criterion_ce(student_logits, labels)
        
        # 组合损失
        total_loss = self.alpha * kd_loss + (1 - self.alpha) * ce_loss
        
        return total_loss

class HybridDistiller(BaseDistiller):
    """中间层+输出层混合蒸馏"""
    def __init__(self, teacher, student, temperature=4.0, 
                 alpha=0.5, beta=0.3):
        super().__init__(teacher, student, temperature, alpha)
        self.beta = beta  # 特征损失权重
        self.criterion_ce = nn.CrossEntropyLoss()
        self.criterion_kl = nn.KLDivLoss(reduction='batchmean')
        
    def compute_kd_loss(self, outputs, labels):
        """
        计算混合蒸馏损失
        outputs: (student_logits, student_features, teacher_logits, teacher_features)
        """
        student_logits, student_feats, teacher_logits, teacher_feats = outputs
        
        # 1. 输出层蒸馏损失
        soft_teacher = F.softmax(teacher_logits / self.temperature, dim=1)
        soft_student = F.log_softmax(student_logits / self.temperature, dim=1)
        kd_loss = self.criterion_kl(soft_student, soft_teacher) * (self.temperature ** 2)
        
        # 2. 特征蒸馏损失
        feature_loss = 0
        for s_feat, t_feat in zip(student_feats, teacher_feats):
            if s_feat.shape != t_feat.shape:
                t_feat = F.adaptive_avg_pool2d(t_feat, s_feat.shape[2:])
            
            # 多种特征损失组合
            mse_loss = F.mse_loss(s_feat, t_feat)
            
            # 注意力转移损失（基于特征图激活）
            s_attention = F.normalize(s_feat.pow(2).mean(1).view(s_feat.size(0), -1))
            t_attention = F.normalize(t_feat.pow(2).mean(1).view(t_feat.size(0), -1))
            attention_loss = F.mse_loss(s_attention, t_attention)
            
            feature_loss += mse_loss + 0.1 * attention_loss
        
        # 3. 硬标签损失
        ce_loss = self.criterion_ce(student_logits, labels)
        
        # 4. 组合所有损失
        total_loss = (self.alpha * kd_loss + 
                     self.beta * feature_loss + 
                     (1 - self.alpha - self.beta) * ce_loss)
        
        return total_loss

# 训练pipeline

In [17]:
# 5_training_pipeline.py
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import yaml
from datetime import datetime

class DistillationPipeline:
    """蒸馏训练流水线"""
    def __init__(self, config_path='config.yaml'):
        self.config = self.load_config(config_path)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
    def load_config(self, config_path):
        """加载配置文件"""
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        return config
        
    def prepare_data(self):
        """准备数据集"""
        # 根据您的数据集修改
        transform = transforms.Compose([
            transforms.Resize((640, 640)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
        # 这里使用示例数据集，请替换为您的数据集
        train_dataset = datasets.FakeData(
            size=1000,
            image_size=(3, 640, 640),
            num_classes=80,
            transform=transform
        )
        
        val_dataset = datasets.FakeData(
            size=200,
            image_size=(3, 640, 640),
            num_classes=80,
            transform=transform
        )
        
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.config['batch_size'],
            shuffle=True,
            num_workers=4
        )
        
        val_loader = DataLoader(
            val_dataset,
            batch_size=self.config['batch_size'],
            shuffle=False,
            num_workers=4
        )
        
        return train_loader, val_loader
    
    def run_distillation(self, method='hybrid'):
        """运行蒸馏训练"""
        print(f"\n{'='*50}")
        print(f"开始 {method} 蒸馏训练")
        print(f"{'='*50}")
        
        # 1. 准备数据
        print("\n[1/4] 准备数据集...")
        train_loader, val_loader = self.prepare_data()
        
        # 2. 加载模型
        print("\n[2/4] 加载教师和学生模型...")
        teacher = self.load_teacher_model()
        student = self.load_student_model()
        
        # 3. 选择蒸馏方法
        print(f"\n[3/4] 初始化 {method} 蒸馏器...")
        distiller = self.get_distiller(method, teacher, student)
        
        # 4. 训练
        print("\n[4/4] 开始训练...")
        train_losses, val_accuracies = distiller.train(
            train_loader,
            val_loader,
            epochs=self.config['epochs'],
            lr=self.config['learning_rate']
        )
        
        # 5. 保存最终模型
        self.save_final_model(student, method)
        
        return train_losses, val_accuracies
    
    def load_teacher_model(self):
        """加载教师模型"""
        teacher = ImprovedYOLO11(self.config['teacher_model_path'])
        teacher.to(self.device)
        return teacher
    
    def load_student_model(self):
        """加载学生模型"""
        # 加载剪枝后的模型
        checkpoint = torch.load(self.config['student_model_path'])
        student = PrunedYOLO11n()
        student.model.load_state_dict(checkpoint['model'])
        student.to(self.device)
        return student
    
    def get_distiller(self, method, teacher, student):
        """获取指定的蒸馏器"""
        config = self.config['distillation'][method]
        
        if method == 'feature':
            return FeatureDistiller(
                teacher, student,
                temperature=config['temperature'],
                alpha=config['alpha']
            )
        elif method == 'output':
            return OutputDistiller(
                teacher, student,
                temperature=config['temperature'],
                alpha=config['alpha']
            )
        elif method == 'hybrid':
            return HybridDistiller(
                teacher, student,
                temperature=config['temperature'],
                alpha=config['alpha'],
                beta=config['beta']
            )
        else:
            raise ValueError(f"未知的蒸馏方法: {method}")
    
    def save_final_model(self, model, method):
        """保存最终蒸馏模型"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        save_path = f"yolo11n_distilled_{method}_{timestamp}.pt"
        
        torch.save({
            'model_state_dict': model.state_dict(),
            'distillation_method': method,
            'config': self.config
        }, save_path)
        
        print(f"\n最终模型已保存至: {save_path}")
        
    def compare_methods(self):
        """比较三种蒸馏方法"""
        methods = ['feature', 'output', 'hybrid']
        results = {}
        
        for method in methods:
            print(f"\n{'='*60}")
            print(f"运行 {method.upper()} 蒸馏方法")
            print(f"{'='*60}")
            
            losses, accs = self.run_distillation(method)
            results[method] = {
                'final_loss': losses[-1],
                'final_accuracy': accs[-1],
                'best_accuracy': max(accs)
            }
        
        # 打印比较结果
        print("\n" + "="*60)
        print("蒸馏方法比较结果")
        print("="*60)
        for method, result in results.items():
            print(f"{method.upper():10} | "
                  f"最终准确率: {result['final_accuracy']:.4f} | "
                  f"最佳准确率: {result['best_accuracy']:.4f} | "
                  f"最终损失: {result['final_loss']:.4f}")
        
        return results

# 配置文件示例
config_yaml = """
# config.yaml
teacher_model_path: "improved_yolo11.pt"
student_model_path: "yolo11n_pruned.pt"

# 训练参数
batch_size: 16
epochs: 50
learning_rate: 0.001

# 蒸馏参数
distillation:
  feature:
    temperature: 4.0
    alpha: 0.7
    
  output:
    temperature: 4.0
    alpha: 0.5
    
  hybrid:
    temperature: 4.0
    alpha: 0.5
    beta: 0.3
"""

# 保存配置文件
with open('config.yaml', 'w') as f:
    f.write(config_yaml)